In [ ]:
import re
import pandas as pd
import tabula
import os
from pypdf import PdfReader
import glob

# SQLAlchemy and Tabulate imports
from sqlalchemy import create_engine
from sqlalchemy.types import Integer, String, Date, BigInteger, JSON
from tabulate import tabulate

# ================================================
# 1. PDF PARSER CLASS
# ================================================

class PDFParser:
    def __init__(self):
        pass
    
    def parse(self, pdf_path):
        """
        Parse a Delivery Instruction (DI) PDF file into a cleaned DataFrame.
        Extracts header info (DI No, Issue Date, Delivery Date, Plant Location)
        and all part details (Part No, Description, Quantities, Kanban Numbers).
        """

        # --- Step 1: Extract table(s) from PDF ---
        tables = tabula.read_pdf(
            pdf_path,
            pages='all',
            multiple_tables=True,
            lattice=False,
            stream=True,
            guess=False,
            pandas_options={'header': None}
        )

        df = pd.concat(tables, ignore_index=True)
        full_text = " ".join(df.astype(str).fillna("").values.flatten())

        # --- Step 2: Extract header fields ---
        di_no = self._find(r"DI\s*No[:.\s]*([0-9]+)", full_text)
        issue_date = self._find(r"Issue\s*Date[:.\s]*([\w-]+)", full_text)
        delivery_date = self._find(r"Delivery\s*Date[:.\s]*([\w-]+)", full_text)
        plant_loc = self._find(r"PLANT\s*(?:LOCATION)?[:.\s-]*([A-Za-z0-9]+)", full_text)
        plant_loc = f"PLANT {plant_loc}" if plant_loc else None

        # --- Step 3: Parse part rows ---
        parts_data = self._extract_parts(df, di_no, issue_date, delivery_date, plant_loc)

        # --- Convert to DataFrame ---
        df_clean = pd.DataFrame(parts_data)

        if df_clean.empty:
            return df_clean

        # --- Clean the DataFrame ---
        df_clean["Part Number"] = df_clean["Part Number"].apply(self._shorten_part_no)
        df_clean["Part Name"] = df_clean["Part Name"].apply(self._clean_part_name)
        df_clean["Kanban Numbers"] = df_clean["Kanban Numbers"].apply(
            lambda nums: [k for k in nums if not str(k).startswith('6000')]
        )
        df_clean["Packaging Qty"] = pd.to_numeric(df_clean["Packaging Qty"], 
            errors="coerce").fillna(0).astype(int)
        df_clean["Number of Cards"] = pd.to_numeric(df_clean["Number of Cards"], 
            errors="coerce").fillna(0).astype(int)

        return df_clean

    # --- Helper methods for parsing and cleaning ---
    def _find(self, pattern, text):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    # --- Method to parse parts from DataFrame ---
    def _extract_parts(self, df, di_no, issue_date, delivery_date, plant_location):
        preferred = re.compile(r"\b[A-Z]-\d{3,4}\b")   # e.g., P-703, H-2704
        fallback = re.compile(r"[A-Z0-9]+-\d+[A-Z0-9-]*")  # e.g., JOON-001, JH-00H5-2729
        kanban_pattern = re.compile(r"\b\d{10}\b")   # 10-digit Kanban numbers
        qty_pattern = re.compile(r"(\d+)\s+(\d+)\s+([\d.]+)")  # Pkg Qty, No of Cards, Total Qty
        unit_pattern = re.compile(r"\bEA\b|\bNO\s*\d+\b", re.IGNORECASE)

        parts_data = []
        current_part, current_desc = None, None
        current_kanbans = []
        pkg_qty, num_cards = None, None

        for _, row in df.iterrows():
            row_text = " ".join(row.astype(str)).strip()

            part_match = preferred.search(row_text) or fallback.search(row_text)

            if part_match:
                # Save previous part before moving to next
                if current_part:
                    parts_data.append(self._assemble_record(
                        di_no, issue_date, delivery_date, plant_location,
                        current_part, current_desc, pkg_qty, num_cards, current_kanbans))

                current_part = part_match.group(0)

                qty_match = qty_pattern.search(row_text)
                
                if qty_match:
                    pkg_qty, num_cards, _ = qty_match.groups()
                else:
                    pkg_qty = num_cards = None

                desc = row_text
                desc = desc.replace(current_part, "")
                desc = qty_pattern.sub("", desc)
                desc = unit_pattern.sub("", desc)
                desc = desc.replace("-", "").strip()

                current_desc = desc
                current_kanbans = []
                continue

            # Detect Kanban numbers
            kanbans = kanban_pattern.findall(row_text)
            if kanbans:
                current_kanbans.extend(kanbans)

        # Append last part
        if current_part:
            parts_data.append(self._assemble_record(
                di_no, issue_date, delivery_date, plant_location,
                current_part, current_desc, pkg_qty, num_cards, current_kanbans
            ))
        return parts_data

    # --- Step 5: Assemble part rows ---
    def _assemble_record(self, di_no, issue_date, delivery_date, plant_location,
                        part, desc, pkg_qty, num_cards, kanbans):
        total = int(pkg_qty) * int(num_cards) if pkg_qty and num_cards else None
        return {
            "DI No": di_no,
            "Issue Date": issue_date,
            "Delivery Date": delivery_date,
            "Plant Location": plant_location,
            "Part Number": part,
            "Part Name": desc,
            "Packaging Qty": pkg_qty,
            "Number of Cards": num_cards,
            "Total Qty": total,
            "Kanban Numbers": kanbans
        }
    
    # --- Cleaning Helpers ---
    def _shorten_part_no(self, pn):
        pn = str(pn)
        parts = pn.split("-")

        if len(parts) >= 2:
            code = parts[-2]
            digits = parts[-1]
            if re.fullmatch(r"\d{3,6}", digits):
                m = re.search(r"[A-Za-z]", code)
                if m:
                    return f"{m.group(0).upper()}-{digits}"
                
        m = re.search(r"([A-Za-z])[-]?(\d{3,6})", pn)
        return f"{m.group(1).upper()}-{m.group(2)}" if m else pn

    def _clean_part_name(self, name):
        n = str(name)
        n = re.sub(r"^\s*\d+\s+", "", n)
        n = re.sub(r"\bJOONHEE\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\bJOON-?\d*\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\bJH\d{1,3}\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\bEA\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\bNO\s*\d+\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\bnan\b", "", n, flags=re.IGNORECASE)
        n = re.sub(r"\s+", " ", n).strip()
        return n


# ================================================
# 2. DATABASE MANAGEMENT
# ================================================

class DatabaseManager:
    def __init__(self):
        self.db_user = input("Enter DB username: ")
        self.db_pass = input("Enter DB password: ")
        self.host = input("Enter DB host (default: localhost): ")
        self.port = input("Enter DB port (default: 4****): ")
        self.database = input("Enter DB name: ")

        self.engine = self._create_engine(
            f"mysql+pymysql://{self.db_user}:{self.db_pass}@"
            f"{self.host}:{self.port}/{self.database}"
        )

        self.existing_records = pd.read_sql("""
            SELECT `DI No`, `Part Number`, `Issue Date`, `Delivery Date`
            FROM delivery_instructions
        """, self.engine)
    
    def _create_engine(self, connection_string):
        try:
            engine = create_engine(connection_string)
            return engine
        except Exception as e:
            print(f"Error creating engine: {e}")
            raise
    
    def _insert_new_records(self, df):
        # Convert date columns to proper date format
        df["Issue Date"] = pd.to_datetime(df["Issue Date"], format="%d-%b-%Y", errors="coerce").dt.date
        df["Delivery Date"] = pd.to_datetime(df["Delivery Date"], format="%d-%b-%Y", errors="coerce").dt.date
        
        df.to_sql(
            name="delivery_instructions",
            con=self.engine,
            if_exists="append",
            index=False,
            dtype={
                "DI No": BigInteger(),
                "Issue Date": Date(),
                "Delivery Date": Date(),
                "Plant Location": String(100),
                "Part Number": String(50),
                "Part Name": String(255),
                "Packaging Qty": Integer(),
                "Number of Cards": Integer(),
                "Total Qty": Integer(),
                "Kanban Numbers": JSON(),
            }
        )


# ================================================
# 3. MAIN PROCESSING PIPELINE & LOGIC
# ================================================

class DIProcessor:
    def __init__(self, pdf_folder):
        self.folder = pdf_folder
        self.parser = PDFParser()
        self.db = DatabaseManager()

    def run(self):
        pdf_files = glob.glob(os.path.join(self.folder, "*.pdf"))
        print(f"\n📁 Found {len(pdf_files)} PDF files in folder.\n")

        # --- Process each PDF ---
        for pdf_path in pdf_files:
            file_name = os.path.basename(pdf_path)
            print(f"\n📄 Processing: {file_name}")

            try:
                # Step 1: Parse PDF
                df_clean = self.parser.parse(pdf_path)

                if df_clean.empty:
                    print("⚠️ No data extracted from this PDF, skipping.")
                    continue

                # Step 2: Extract DI No
                di_no = str(df_clean["DI No"].iloc[0]).strip()

                # Step 3: Check if DI No already exists in database
                if di_no in self.db.existing_records["DI No"].astype(str).values:
                    print(f"⚠️ DI No {di_no} already exists in the database. Skipping insertion.")
                    continue
                else:
                    print(f"✅ DI No {di_no} not found in database. Checking for part-level duplicates...")

                # Step 4: Ensure type consistency
                df_clean["DI No"] = df_clean["DI No"].astype(str)
                df_clean["Part Number"] = df_clean["Part Number"].astype(str)
                self.db.existing_records["DI No"] = self.db.existing_records["DI No"].astype(str)
                self.db.existing_records["Part Number"] = self.db.existing_records["Part Number"].astype(str)

                # Step 5: Compare for duplicates (DI No + Part Number)
                comparison = df_clean.merge(
                    self.db.existing_records,
                    on=["DI No", "Part Number"],
                    how="left",
                    indicator=True
                )

                dupes = comparison[comparison["_merge"] == "both"]
                new_records = comparison[comparison["_merge"] == "left_only"].drop(columns="_merge")

                # Rename columns back to their original names
                rename_map = {
                    "Issue Date_x": "Issue Date",
                    "Delivery Date_x": "Delivery Date",
                }
                new_records = new_records.rename(columns=rename_map)

                # Drop leftover _y columns if they exist
                new_records = new_records[[col for col in new_records.columns if not col.endswith("_y")]]

                # Step 6: Handle duplicates or insert new data
                if not dupes.empty:
                    print(f"⚠️ Found {len(dupes)} duplicate records based on DI No and Part Number:")
                    print(tabulate(dupes[["DI No", "Part Number"]], headers="keys", tablefmt="psql"))
                    print("🚫 Skipping insert for these duplicates.")

                elif not new_records.empty:
                    print(f"✅ No duplicates found. Inserting {len(new_records)} new records...")
                    self.db._insert_new_records(new_records)
                    print("✅ Successfully inserted into MySQL.")

                    # Update local database cache (so we don't reprocess next file)
                    self.db.existing_records = pd.concat(
                        [self.db.existing_records, new_records[["DI No", "Part Number", "Issue Date", "Delivery Date"]]],
                        ignore_index=True
                    )
                else:
                    print("⚠️ No new records to insert.")

            except Exception as e:
                print(f"❌ Error processing {file_name}: {e}")
                import traceback
                traceback.print_exc()

        print("\n🎯 Processing complete for all PDFs.")


# ================================================
# 4. RUN THE PROCESSOR
# ================================================

def select_folder():
    """
    Opens a GUI file dialog to select the PDF folder.
    """
    try:
        import tkinter as tk
        from tkinter import filedialog
        
        print("\n" + "="*60)
        print("📁 PDF FOLDER SELECTION")
        print("="*60)
        
        root = tk.Tk()
        root.withdraw()  # Hide the main window
        root.attributes('-topmost', True)  # Bring dialog to front
        
        print("\n🖱️  Opening file dialog... (Check your taskbar if you don't see it)")
        folder = filedialog.askdirectory(title="Select PDF Folder")
        root.destroy()
        
        if not folder:
            print("❌ No folder selected. Exiting...")
            return None
        
        # Validate folder exists (should already exist if selected via dialog, but good practice)
        if not os.path.exists(folder) or not os.path.isdir(folder):
            print(f"\n❌ Error: Invalid folder: {folder}")
            return None
        
        # Count PDF files
        pdf_count = len(glob.glob(os.path.join(folder, "*.pdf")))
        print(f"\n✅ Selected folder: {folder}")
        print(f"📄 Found {pdf_count} PDF file(s) in this folder.")
        
        if pdf_count == 0:
            print("\n⚠️  Warning: No PDF files found in this folder!")
            proceed = input("Continue anyway? (y/n): ").strip().lower()
            if proceed != 'y':
                print("❌ Operation cancelled.")
                return None
        
        return folder
        
    except ImportError:
        print("❌ Error: tkinter not available.")
        print("Please install tkinter or use manual path entry.")
        return None
    except Exception as e:
        print(f"❌ Error opening file dialog: {e}")
        return None


if __name__ == "__main__":
    print("\n" + "="*60)
    print("🤖 DELIVERY INSTRUCTION PDF PROCESSOR")
    print("="*60)
    
    folder = select_folder()
    
    if folder is None:
        print("\n❌ No folder selected. Exiting program.")
        input("\nPress Enter to exit...")
        exit(0)
    
    print("\n" + "="*60)
    print("🚀 STARTING PROCESSING...")
    print("="*60)
    
    processor = DIProcessor(folder)
    processor.run()
    
    print("\n" + "="*60)
    print("✅ PROCESSING COMPLETE!")
    print("="*60)
    



🤖 DELIVERY INSTRUCTION PDF PROCESSOR

📁 PDF FOLDER SELECTION

🖱️  Opening file dialog... (Check your taskbar if you don't see it)

✅ Selected folder: C:/Users/User/Desktop/Python/Project/AISB/5. Sales as per DI/11 - November
📄 Found 58 PDF file(s) in this folder.

🚀 STARTING PROCESSING...

📁 Found 58 PDF files in folder.


📄 Processing: 6000165960.pdf
⚠️ DI No 6000165960 already exists in the database. Skipping insertion.

📄 Processing: 6000165961.pdf
⚠️ DI No 6000165961 already exists in the database. Skipping insertion.

📄 Processing: 6000165997.pdf
⚠️ DI No 6000165997 already exists in the database. Skipping insertion.

📄 Processing: 6000166009.pdf
⚠️ DI No 6000166009 already exists in the database. Skipping insertion.

📄 Processing: 6000166087.pdf
⚠️ DI No 6000166087 already exists in the database. Skipping insertion.

📄 Processing: 6000166127.pdf
⚠️ DI No 6000166127 already exists in the database. Skipping insertion.

📄 Processing: 6000166128.pdf
⚠️ DI No 6000166128 already exist